# 05 · Test, dibuka **sekali**
Fit pada Train+Val, prediksi Test. Setelah ini tidak ada perubahan pipeline. Analisis tambahan = "eksploratif".

In [ ]:
import json
import numpy as np, pandas as pd
from utils import *

flag = PROC / 'test_opened.flag'
assert not flag.exists(), 'Test sudah dibuka. Hapus flag hanya untuk analisis eksploratif.'

df, price = load()
cols = [c for c in df.columns if c not in ('target', 'target2', 'target0', 'split')]
base = df[df.split.isin(['train', 'val', 'test']) & df.p_up.notna()]
trv, te = base[base.split != 'test'], base[base.split == 'test']
y, V = te.target.values, variants(cols, price)
fin, gate = json.load(open(PROC / 'finalist.json'))['finalist'], json.load(open(PROC / 'gate.json'))
print('train+val', len(trv), '| test', len(te), '| finalis:', fin, '| gate:', gate['gate'])

In [ ]:
P = {n: fit_predict(trv[u], trv.target, te[u], k) for n, (k, u) in V.items()}
maj = 1 if (trv.target > 0).mean() > .5 else -1
rows = [{'model': n + (' *' if n == fin else ''), **evaluate(y, p, trv.target.std())} for n, p in P.items()]
rows += [{'model': 'zero', **evaluate(y, 0 * y)}, {'model': 'kemarin', **evaluate(y, te.ret_lag0.values)}, {'model': 'mayoritas', 'DirAcc': np.mean(np.sign(y) == maj)}]
print(pd.DataFrame(rows).set_index('model').round(5).to_string())

In [ ]:
z = 0 * y
for n in V:
    if n == 'A': continue
    s0, p0 = dm_test(y, z, P[n]); sA, pA = dm_test(y, P['A'], P[n])
    lo, hi = rmse_diff_ci(y, P[n], P['A'])
    print(f'{n}: DM vs zero p={p0:.3f} | DM vs A p={pA:.3f} | RMSE({n})-RMSE(A) = {np.sqrt(np.mean((y-P[n])**2)) - np.sqrt(np.mean((y-P["A"])**2)):+.6f} [{lo:+.6f}, {hi:+.6f}]')
auc, (lo, hi) = auc_ci((y > 0).astype(int), te.p_up.values)
print(f'\np_up di Test: AUC={auc:.3f} [{lo:.3f}, {hi:.3f}] | n={len(te)} hari (kecil, jangan tarik kesimpulan kausal)')
flag.write_text('opened')